In [1]:
import torch
import torch.nn as nn
from torch import tensor

### Use Simple Embedding

In [2]:
embedding = nn.Embedding(5, 10)

In [3]:
embedding.weight

Parameter containing:
tensor([[ 0.4790, -0.5685, -0.5136,  0.3194, -1.9425,  0.7368,  1.4662,  0.6902,
         -1.4315, -0.5876],
        [-1.5715,  1.1445, -0.6235,  0.7130,  0.7996,  0.3189,  0.7199, -1.3828,
          1.3479,  0.4245],
        [ 0.5619, -0.2065, -0.4304, -0.6097,  0.5258, -1.4456,  0.8899, -0.7702,
         -0.4146,  0.1535],
        [ 0.3775, -0.8393, -0.2631,  0.2442,  0.9337,  0.3465,  2.6458, -0.2071,
          2.2588, -0.8580],
        [ 0.5070,  0.5101,  0.9698, -0.6185,  0.8607, -1.0275, -0.0179, -0.3508,
         -1.8279, -0.1002]], requires_grad=True)

In [6]:
embedding.weight.shape

torch.Size([5, 10])

In [5]:
nn.init.uniform_(embedding.weight, -0.1, 0.1)
embedding.weight

Parameter containing:
tensor([[-0.0108, -0.0071,  0.0917, -0.0441,  0.0387, -0.0280, -0.0790,  0.0882,
          0.0510,  0.0979],
        [-0.0725,  0.0407, -0.0543,  0.0699,  0.0804, -0.0056,  0.0716, -0.0382,
         -0.0038, -0.0821],
        [-0.0864,  0.0414, -0.0843, -0.0794,  0.0647, -0.0931, -0.0937, -0.0654,
          0.0794, -0.0789],
        [ 0.0856,  0.0007, -0.0356,  0.0036,  0.0791,  0.0076,  0.0894,  0.0015,
         -0.0510, -0.0550],
        [-0.0654,  0.0693, -0.0470,  0.0451,  0.0907,  0.0392, -0.0579,  0.0297,
          0.0056, -0.0450]], requires_grad=True)

In [7]:
# 5 Token
x = torch.tensor([0, 1, 2, 3, 4])

y = embedding(x)
y.shape

torch.Size([5, 10])

In [8]:
y

tensor([[-0.0108, -0.0071,  0.0917, -0.0441,  0.0387, -0.0280, -0.0790,  0.0882,
          0.0510,  0.0979],
        [-0.0725,  0.0407, -0.0543,  0.0699,  0.0804, -0.0056,  0.0716, -0.0382,
         -0.0038, -0.0821],
        [-0.0864,  0.0414, -0.0843, -0.0794,  0.0647, -0.0931, -0.0937, -0.0654,
          0.0794, -0.0789],
        [ 0.0856,  0.0007, -0.0356,  0.0036,  0.0791,  0.0076,  0.0894,  0.0015,
         -0.0510, -0.0550],
        [-0.0654,  0.0693, -0.0470,  0.0451,  0.0907,  0.0392, -0.0579,  0.0297,
          0.0056, -0.0450]], grad_fn=<EmbeddingBackward0>)

In [9]:
# 1 Token
x = torch.tensor([0])

y = embedding(x)
y.shape

torch.Size([1, 10])

In [10]:
y, embedding.weight[0]

(tensor([[-0.0108, -0.0071,  0.0917, -0.0441,  0.0387, -0.0280, -0.0790,  0.0882,
           0.0510,  0.0979]], grad_fn=<EmbeddingBackward0>),
 tensor([-0.0108, -0.0071,  0.0917, -0.0441,  0.0387, -0.0280, -0.0790,  0.0882,
          0.0510,  0.0979], grad_fn=<SelectBackward0>))

In [11]:
# out of index
x = torch.tensor([5])

y = embedding(x)

IndexError: index out of range in self

### Sequential

In [12]:
embedding = nn.Embedding(5, 10)

In [13]:
embedding.weight.requires_grad

True

In [14]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(5, 10)
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)
        x = x.sum(dim=1)
        x = x.unsqueeze(1)
        return self.linear(x)

In [15]:
model = MyModel()

In [16]:
x = torch.tensor([
    [0],
])

In [17]:
y = model(x)
y

tensor([[0.9161]], grad_fn=<AddmmBackward0>)

In [18]:
x = torch.tensor([
    [1],
])

In [19]:
y = model(x)
y

tensor([[-0.1187]], grad_fn=<AddmmBackward0>)

In [20]:
x = torch.tensor([
    [0, 1],
])
y = model(x)
y

tensor([[0.3987]], grad_fn=<AddmmBackward0>)

In [21]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

In [22]:
loss_fn = nn.MSELoss()

In [23]:
y_true = tensor([[1.]])
loss = loss_fn(y, y_true)
loss

tensor(0.3616, grad_fn=<MseLossBackward0>)

In [24]:
loss.backward()

In [25]:
model.embedding.weight

Parameter containing:
tensor([[ 1.1298,  0.7595,  0.3752, -0.0864, -0.6699, -0.9884, -0.5362,  1.1408,
         -0.5677, -0.9480],
        [-0.5033, -0.5263,  0.8064, -0.1119,  0.5209,  0.3578,  0.2971,  0.5531,
          0.3468, -0.5854],
        [ 0.0040,  0.0815, -0.4085, -0.5931,  1.4197, -0.0035, -1.5460, -0.5058,
         -1.2837,  0.0869],
        [-0.9523,  0.0224,  1.3417,  1.0568,  0.1274,  0.7215, -0.9000,  0.3316,
         -0.4865,  0.8704],
        [-1.3235, -0.5862,  1.1491, -1.2433, -0.8655, -1.6547, -1.7745,  1.4654,
         -0.7053,  2.4943]], requires_grad=True)

In [26]:
model.embedding.weight.grad

tensor([[0.4023, 0.4023, 0.4023, 0.4023, 0.4023, 0.4023, 0.4023, 0.4023, 0.4023,
         0.4023],
        [0.4023, 0.4023, 0.4023, 0.4023, 0.4023, 0.4023, 0.4023, 0.4023, 0.4023,
         0.4023],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])

In [27]:
optimizer.step()

In [28]:
model.embedding.weight

Parameter containing:
tensor([[ 1.1257,  0.7555,  0.3711, -0.0905, -0.6739, -0.9925, -0.5402,  1.1368,
         -0.5717, -0.9520],
        [-0.5074, -0.5303,  0.8024, -0.1159,  0.5168,  0.3538,  0.2931,  0.5491,
          0.3427, -0.5894],
        [ 0.0040,  0.0815, -0.4085, -0.5931,  1.4197, -0.0035, -1.5460, -0.5058,
         -1.2837,  0.0869],
        [-0.9523,  0.0224,  1.3417,  1.0568,  0.1274,  0.7215, -0.9000,  0.3316,
         -0.4865,  0.8704],
        [-1.3235, -0.5862,  1.1491, -1.2433, -0.8655, -1.6547, -1.7745,  1.4654,
         -0.7053,  2.4943]], requires_grad=True)